**Check the Kaggle environment**

In [32]:
import os
import torch

print("=" * 70)
print("KAGGLE ENVIRONMENT CHECK")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nInput datasets:")
for item in os.listdir("/kaggle/input"):
    print(" -", item)

print("\nEnvironment check complete.")

KAGGLE ENVIRONMENT CHECK
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4

Input datasets:
 - datasets

Environment check complete.


In [33]:
import os

root = "/kaggle/input/datasets"

for item in os.listdir(root):
    path = os.path.join(root, item)
    print(f"{item}  ->  {'DIR' if os.path.isdir(path) else 'FILE'}")

mahmudnadim  ->  DIR


In [34]:
import os

root = "/kaggle/input/datasets/mahmudnadim"

for item in os.listdir(root):
    path = os.path.join(root, item)
    print(f"{item}  ->  {'DIR' if os.path.isdir(path) else 'FILE'}")

robust-crop-disease-plantvillage  ->  DIR
robust-crop-disease-checkpoint  ->  DIR
robust-crop-disease-splits  ->  DIR


**Inspect their contents**

In [35]:
import os

root = "/kaggle/input/datasets/mahmudnadim/robust-crop-disease-splits"

for current_root, dirs, files in os.walk(root):
    print("\nPATH:", current_root)
    for f in files:
        print("  -", f)


PATH: /kaggle/input/datasets/mahmudnadim/robust-crop-disease-splits
  - train_validation_split.npz

PATH: /kaggle/input/datasets/mahmudnadim/robust-crop-disease-splits/splits-20260929T040001Z-1-001

PATH: /kaggle/input/datasets/mahmudnadim/robust-crop-disease-splits/splits-20260929T040001Z-1-001/splits
  - color_test.txt
  - color_train.txt


In [37]:
# Verify the exact original PlantVillage split files

SPLITS_DIR = "/kaggle/input/datasets/mahmudnadim/robust-crop-disease-splits/splits-20260929T040001Z-1-001/splits"

TRAIN_SPLIT = os.path.join(SPLITS_DIR, "color_train.txt")
TEST_SPLIT = os.path.join(SPLITS_DIR, "color_test.txt")

print("Train split exists:", os.path.exists(TRAIN_SPLIT))
print("Test split exists:", os.path.exists(TEST_SPLIT))

if os.path.exists(TRAIN_SPLIT):
    print("Train images:", sum(1 for _ in open(TRAIN_SPLIT, "r", encoding="utf-8")))

if os.path.exists(TEST_SPLIT):
    print("Test images:", sum(1 for _ in open(TEST_SPLIT, "r", encoding="utf-8")))

Train split exists: True
Test split exists: True
Train images: 43596
Test images: 10709


In [39]:
import numpy as np

SPLIT_NPZ = "/kaggle/input/datasets/mahmudnadim/robust-crop-disease-splits/train_validation_split.npz"

split_data = np.load(SPLIT_NPZ)

print("Keys:", split_data.files)

for key in split_data.files:
    arr = split_data[key]
    print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

Keys: ['train_idx', 'val_idx']
train_idx: shape=(39207,), dtype=int64
val_idx: shape=(4389,), dtype=int64


In [40]:
# Reconstruct the exact train/validation/test file lists

with open(TRAIN_SPLIT, "r", encoding="utf-8") as f:
    original_train_paths = [line.strip() for line in f if line.strip()]

with open(TEST_SPLIT, "r", encoding="utf-8") as f:
    test_paths = [line.strip() for line in f if line.strip()]

train_idx = split_data["train_idx"]
val_idx = split_data["val_idx"]

train_paths = [original_train_paths[i] for i in train_idx]
val_paths = [original_train_paths[i] for i in val_idx]

print("Reconstructed split:")
print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))

print("\nFirst train:", train_paths[0])
print("First validation:", val_paths[0])
print("First test:", test_paths[0])

Reconstructed split:
Train: 39207
Validation: 4389
Test: 10709

First train: raw/color/Raspberry___healthy/a37f34f7-022d-461a-8a3d-95f5cd774e35___Mary_HL 9155.JPG
First validation: raw/color/Raspberry___healthy/67c481c3-711a-41d5-b818-73aab9280294___Mary_HL 9228.JPG
First test: raw/color/Raspberry___healthy/6c2e049f-38c9-42c5-864f-613e6e12d59e___Mary_HL 6318.JPG


In [41]:
# Verify reconstructed paths against the Kaggle PlantVillage dataset

def resolve_pv_path(relative_path):
    return os.path.join(
        PLANTVILLAGE_DIR,
        relative_path.replace("raw/color/", "", 1)
    )

sample_paths = train_paths[:3] + val_paths[:3] + test_paths[:3]

resolved = [resolve_pv_path(p) for p in sample_paths]

print("Checking", len(resolved), "sample paths...")

for p in resolved:
    print(os.path.exists(p), "->", p)

print("\nAll sample paths valid:", all(os.path.exists(p) for p in resolved))

Checking 9 sample paths...
True -> /kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Raspberry___healthy/a37f34f7-022d-461a-8a3d-95f5cd774e35___Mary_HL 9155.JPG
True -> /kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Raspberry___healthy/368f3c83-5bd2-4906-b418-57b6ceff9bfe___Mary_HL 9160.JPG
True -> /kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Raspberry___healthy/98226f66-15a1-40d5-9b62-04ae5b4fd8d5___Mary_HL 9159.JPG
True -> /kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Raspberry___healthy/67c481c3-711a-41d5-b818-73aab9280294___Mary_HL 9228.JPG
True -> /kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Raspberry___healthy/c1311840-6ce5-4d86-a282-b483d67a0155___Mary_HL 9227.JPG
True -> /kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Raspberry___healthy/afe05bf6-ccef-4665-8a62-6a4a4c2d2939___Mary_HL 9239.JPG
True -> /kaggle/input/dat

In [42]:
# Verify the 38 PlantVillage classes

class_names = sorted([
    d for d in os.listdir(PLANTVILLAGE_DIR)
    if os.path.isdir(os.path.join(PLANTVILLAGE_DIR, d))
])

print("Number of classes:", len(class_names))

for i, name in enumerate(class_names):
    print(f"{i:02d}: {name}")

Number of classes: 38
00: Apple___Apple_scab
01: Apple___Black_rot
02: Apple___Cedar_apple_rust
03: Apple___healthy
04: Blueberry___healthy
05: Cherry_(including_sour)___Powdery_mildew
06: Cherry_(including_sour)___healthy
07: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
08: Corn_(maize)___Common_rust_
09: Corn_(maize)___Northern_Leaf_Blight
10: Corn_(maize)___healthy
11: Grape___Black_rot
12: Grape___Esca_(Black_Measles)
13: Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
14: Grape___healthy
15: Orange___Haunglongbing_(Citrus_greening)
16: Peach___Bacterial_spot
17: Peach___healthy
18: Pepper,_bell___Bacterial_spot
19: Pepper,_bell___healthy
20: Potato___Early_blight
21: Potato___Late_blight
22: Potato___healthy
23: Raspberry___healthy
24: Soybean___healthy
25: Squash___Powdery_mildew
26: Strawberry___Leaf_scorch
27: Strawberry___healthy
28: Tomato___Bacterial_spot
29: Tomato___Early_blight
30: Tomato___Late_blight
31: Tomato___Leaf_Mold
32: Tomato___Septoria_leaf_spot
33: Tomato___S

In [44]:
# Build and verify the exact 38-class label mapping

class_to_idx = {name: idx for idx, name in enumerate(class_names)}

def get_label_from_path(relative_path):
    class_name = relative_path.replace("\\", "/").split("/")[2]
    return class_to_idx[class_name]

train_labels = [get_label_from_path(p) for p in train_paths]
val_labels = [get_label_from_path(p) for p in val_paths]
test_labels = [get_label_from_path(p) for p in test_paths]

print("Train labels:", len(train_labels))
print("Validation labels:", len(val_labels))
print("Test labels:", len(test_labels))

print("\nUnique classes:")
print("Train:", len(set(train_labels)))
print("Validation:", len(set(val_labels)))
print("Test:", len(set(test_labels)))

print("\nLabel range:")
print(min(train_labels), "to", max(train_labels))

print("\nLabel mapping verified:", set(train_labels) == set(range(38)))

Train labels: 39207
Validation labels: 4389
Test labels: 10709

Unique classes:
Train: 38
Validation: 38
Test: 38

Label range:
0 to 37

Label mapping verified: True


In [45]:
# Corruption-aware training configuration

IMAGE_SIZE = 224
CORRUPTION_PROBABILITY = 0.5

print("Image size:", IMAGE_SIZE)
print("Corruption probability:", CORRUPTION_PROBABILITY)
print("Corruptions:")
print("  - Gaussian blur")
print("  - Motion blur")
print("  - JPEG compression")
print("  - Brightness")
print("  - Gaussian noise")

Image size: 224
Corruption probability: 0.5
Corruptions:
  - Gaussian blur
  - Motion blur
  - JPEG compression
  - Brightness
  - Gaussian noise


In [46]:
# Check whether the required transform/corruption definitions already exist

required_names = [
    "gaussian_blur",
    "motion_blur",
    "jpeg_compression",
    "brightness",
    "gaussian_noise",
]

for name in required_names:
    print(f"{name}: {'DEFINED' if name in globals() else 'NOT DEFINED'}")

gaussian_blur: NOT DEFINED
motion_blur: NOT DEFINED
jpeg_compression: NOT DEFINED
brightness: NOT DEFINED
gaussian_noise: NOT DEFINED


In [47]:
# Exact corruption functions from the original experiment

import random
import io

from PIL import Image, ImageEnhance, ImageFilter
from torchvision import transforms

def gaussian_blur(image):
    radius = random.uniform(0.4, 1.2)
    return image.filter(ImageFilter.GaussianBlur(radius=radius))


def motion_blur(image):
    kernel_size = random.choice([3, 5, 7])

    arr = np.asarray(image).astype(np.float32)

    pad = kernel_size // 2
    padded = np.pad(
        arr,
        ((0, 0), (pad, pad), (0, 0)),
        mode="edge"
    )

    blurred = np.zeros_like(arr)

    for i in range(kernel_size):
        blurred += padded[:, i:i + arr.shape[1], :]

    blurred /= kernel_size

    blurred = np.clip(blurred, 0, 255).astype(np.uint8)

    return Image.fromarray(blurred)


def jpeg_compression(image):
    quality = random.randint(45, 75)

    buffer = io.BytesIO()

    image.save(
        buffer,
        format="JPEG",
        quality=quality
    )

    buffer.seek(0)

    return Image.open(buffer).convert("RGB")


def brightness_variation(image):
    factor = random.uniform(0.65, 1.35)

    return ImageEnhance.Brightness(image).enhance(factor)


def gaussian_noise(image):
    image_array = (
        np.asarray(image)
        .astype(np.float32)
        / 255.0
    )

    noise_std = random.uniform(0.02, 0.06)

    noise = np.random.normal(
        0.0,
        noise_std,
        size=image_array.shape
    )

    noisy = np.clip(
        image_array + noise,
        0.0,
        1.0
    )

    noisy = (noisy * 255.0).astype(np.uint8)

    return Image.fromarray(noisy)


CORRUPTION_TYPES = [
    "gaussian_blur",
    "motion_blur",
    "jpeg_compression",
    "brightness",
    "gaussian_noise"
]

print("Corruption functions restored.")
print("Types:", CORRUPTION_TYPES)

Corruption functions restored.
Types: ['gaussian_blur', 'motion_blur', 'jpeg_compression', 'brightness', 'gaussian_noise']


In [49]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print("ImageNet normalization constants restored.")
print("Mean:", IMAGENET_MEAN)
print("Std :", IMAGENET_STD)

ImageNet normalization constants restored.
Mean: [0.485, 0.456, 0.406]
Std : [0.229, 0.224, 0.225]


In [50]:
def apply_random_corruption(image):
    if random.random() >= CORRUPTION_PROBABILITY:
        return image

    corruption = random.choice(CORRUPTION_TYPES)

    if corruption == "gaussian_blur":
        image = gaussian_blur(image)

    elif corruption == "motion_blur":
        image = motion_blur(image)

    elif corruption == "jpeg_compression":
        image = jpeg_compression(image)

    elif corruption == "brightness":
        image = brightness_variation(image)

    elif corruption == "gaussian_noise":
        image = gaussian_noise(image)

    return image


corruption_aware_train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Lambda(apply_random_corruption),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


corruption_aware_eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


print("Random corruption pipeline: READY")
print("Training transform: READY")
print("Evaluation transform: READY")
print("Evaluation corruption: NONE")

Random corruption pipeline: READY
Training transform: READY
Evaluation transform: READY
Evaluation corruption: NONE


In [51]:
# Build Kaggle-local dataset records

train_records = [
    {
        "path": resolve_pv_path(p),
        "label": label
    }
    for p, label in zip(train_paths, train_labels)
]

val_records = [
    {
        "path": resolve_pv_path(p),
        "label": label
    }
    for p, label in zip(val_paths, val_labels)
]

test_records = [
    {
        "path": resolve_pv_path(p),
        "label": label
    }
    for p, label in zip(test_paths, test_labels)
]

print("Records created:")
print("Train:", len(train_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))

print("\nSample record:")
print(train_records[0])

Records created:
Train: 39207
Validation: 4389
Test: 10709

Sample record:
{'path': '/kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Raspberry___healthy/a37f34f7-022d-461a-8a3d-95f5cd774e35___Mary_HL 9155.JPG', 'label': 23}


In [53]:
# ============================================================
# RESTORE CORE IMPORTS + DATASET CLASS
# ============================================================

import os
import io
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image, ImageEnhance, ImageFilter

from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("CORE ENVIRONMENT RESTORED")
print("=" * 70)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
print("Device  :", DEVICE)


# ============================================================
# PlantVillage Dataset
# ============================================================

class PlantVillageDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["path"]
        label = int(row["label"])

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label


print("\nPlantVillageDataset restored: PASS")
print("=" * 70)

CORE ENVIRONMENT RESTORED
PyTorch : 2.10.0+cu128
CUDA    : True
Device  : cuda

PlantVillageDataset restored: PASS


In [54]:
# ============================================================
# CREATE CORRUPTION-AWARE DATASETS
# ============================================================

corruption_train_dataset = PlantVillageDataset(
    dataframe=pd.DataFrame(train_records),
    transform=corruption_aware_train_transform
)

corruption_val_dataset = PlantVillageDataset(
    dataframe=pd.DataFrame(val_records),
    transform=corruption_aware_eval_transform
)

corruption_test_dataset = PlantVillageDataset(
    dataframe=pd.DataFrame(test_records),
    transform=corruption_aware_eval_transform
)

print("=" * 70)
print("CORRUPTION-AWARE DATASETS CREATED")
print("=" * 70)

print("Train      :", len(corruption_train_dataset))
print("Validation :", len(corruption_val_dataset))
print("Test       :", len(corruption_test_dataset))

assert len(corruption_train_dataset) == 39207
assert len(corruption_val_dataset) == 4389
assert len(corruption_test_dataset) == 10709

print("\nDataset size verification: PASS")

CORRUPTION-AWARE DATASETS CREATED
Train      : 39207
Validation : 4389
Test       : 10709

Dataset size verification: PASS


In [55]:
# ============================================================
# CREATE DATALOADERS
# ============================================================

BATCH_SIZE = 32
NUM_WORKERS = 2

corruption_train_loader = DataLoader(
    corruption_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

corruption_val_loader = DataLoader(
    corruption_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

corruption_test_loader = DataLoader(
    corruption_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

print("=" * 70)
print("DATALOADERS CREATED")
print("=" * 70)

print("Batch size :", BATCH_SIZE)
print("Workers    :", NUM_WORKERS)

print("\nNumber of batches:")
print("Train      :", len(corruption_train_loader))
print("Validation :", len(corruption_val_loader))
print("Test       :", len(corruption_test_loader))

DATALOADERS CREATED
Batch size : 32
Workers    : 2

Number of batches:
Train      : 1226
Validation : 138
Test       : 335


In [56]:
# ============================================================
# BATCH SANITY CHECK
# ============================================================

images, labels = next(iter(corruption_train_loader))

print("=" * 70)
print("BATCH SANITY CHECK")
print("=" * 70)

print("Images shape :", images.shape)
print("Labels shape :", labels.shape)
print("Images dtype :", images.dtype)
print("Labels dtype :", labels.dtype)
print("Image device :", images.device)
print("Label range  :", labels.min().item(), "to", labels.max().item())

assert images.shape == (32, 3, 224, 224)
assert labels.shape == (32,)
assert images.dtype == torch.float32
assert labels.dtype == torch.int64
assert labels.min() >= 0
assert labels.max() < 38

print("\nBatch verification: PASS")

BATCH SANITY CHECK
Images shape : torch.Size([32, 3, 224, 224])
Labels shape : torch.Size([32])
Images dtype : torch.float32
Labels dtype : torch.int64
Image device : cpu
Label range  : 2 to 37

Batch verification: PASS


In [57]:
# ============================================================
# RESTORE EFFICIENTNETV2-S MODEL
# ============================================================

NUM_CLASSES = 38

model = models.efficientnet_v2_s(weights=None)

# Replace the original classifier with the 38-class head
in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, NUM_CLASSES)
)

# Freeze backbone
for param in model.features.parameters():
    param.requires_grad = False

# Keep classification head trainable
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("=" * 70)
print("EFFICIENTNETV2-S RESTORED")
print("=" * 70)

print("Classes          :", NUM_CLASSES)
print("Input size       :", "224 x 224")
print("Dropout          :", 0.2)
print("Total parameters :", f"{total_params:,}")
print("Trainable        :", f"{trainable_params:,}")
print("Device           :", DEVICE)

EFFICIENTNETV2-S RESTORED
Classes          : 38
Input size       : 224 x 224
Dropout          : 0.2
Total parameters : 20,226,166
Trainable        : 48,678
Device           : cuda


In [58]:
# ============================================================
# RESTORE EPOCH 2 CHECKPOINT
# ============================================================

BEST_CHECKPOINT = (
    "/kaggle/input/datasets/mahmudnadim/"
    "robust-crop-disease-checkpoint/best_model.pth"
)

LATEST_CHECKPOINT = (
    "/kaggle/input/datasets/mahmudnadim/"
    "robust-crop-disease-checkpoint/latest_checkpoint.pth"
)

print("Latest checkpoint exists:", os.path.exists(LATEST_CHECKPOINT))
print("Best checkpoint exists  :", os.path.exists(BEST_CHECKPOINT))

checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=DEVICE
)

print("\nCheckpoint keys:")
print(list(checkpoint.keys()))

Latest checkpoint exists: True
Best checkpoint exists  : True

Checkpoint keys:
['epoch', 'model_state_dict', 'optimizer_state_dict', 'history', 'best_val_macro_f1', 'best_epoch', 'early_stop_counter', 'experiment_config']


In [59]:
# ============================================================
# LOAD EPOCH 2 TRAINING STATE
# ============================================================

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001,
    weight_decay=0.0001
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"] + 1
best_val_macro_f1 = checkpoint["best_val_macro_f1"]
best_epoch = checkpoint["best_epoch"]
early_stop_counter = checkpoint["early_stop_counter"]
history = checkpoint["history"]

print("=" * 70)
print("CHECKPOINT RESTORED")
print("=" * 70)

print("Checkpoint epoch       :", checkpoint["epoch"])
print("Next epoch             :", start_epoch)
print("Best epoch             :", best_epoch)
print("Best Validation Macro F1:", best_val_macro_f1)
print("Early-stop counter     :", early_stop_counter)
print("History epochs         :", len(history))

print("\nModel state: PASS")
print("Optimizer state: PASS")

CHECKPOINT RESTORED
Checkpoint epoch       : 2
Next epoch             : 3
Best epoch             : 2
Best Validation Macro F1: 0.8552633281812322
Early-stop counter     : 0
History epochs         : 6

Model state: PASS
Optimizer state: PASS


In [60]:
# ============================================================
# VERIFY RESTORED HISTORY STRUCTURE
# ============================================================

print("History type:", type(history))

if isinstance(history, dict):
    print("History keys:", list(history.keys()))

    for key, value in history.items():
        try:
            print(f"{key}: {len(value)} entries")
        except TypeError:
            print(f"{key}: {type(value).__name__}")
else:
    print("History length:", len(history))
    print("First entry:", history[0] if len(history) > 0 else "EMPTY")

History type: <class 'dict'>
History keys: ['train_loss', 'val_loss', 'train_accuracy', 'val_accuracy', 'train_macro_f1', 'val_macro_f1']
train_loss: 2 entries
val_loss: 2 entries
train_accuracy: 2 entries
val_accuracy: 2 entries
train_macro_f1: 2 entries
val_macro_f1: 2 entries


In [61]:
# ============================================================
# VERIFY EPOCH 1–2 HISTORY
# ============================================================

for epoch in range(2):
    print(
        f"Epoch {epoch + 1}: "
        f"Train Loss={history['train_loss'][epoch]:.6f}, "
        f"Val Loss={history['val_loss'][epoch]:.6f}, "
        f"Train Acc={history['train_accuracy'][epoch]:.6f}, "
        f"Val Acc={history['val_accuracy'][epoch]:.6f}, "
        f"Train Macro F1={history['train_macro_f1'][epoch]:.6f}, "
        f"Val Macro F1={history['val_macro_f1'][epoch]:.6f}"
    )

assert len(history["train_loss"]) == 2
assert len(history["val_loss"]) == 2
assert len(history["train_macro_f1"]) == 2
assert checkpoint["epoch"] == 2

print("\nCheckpoint history verification: PASS")
print("Ready to resume from Epoch 3.")

Epoch 1: Train Loss=1.058336, Val Loss=0.562105, Train Acc=0.740021, Val Acc=0.860560, Train Macro F1=0.675839, Val Macro F1=0.823893
Epoch 2: Train Loss=0.593663, Val Loss=0.422590, Train Acc=0.828959, Val Acc=0.882206, Train Macro F1=0.790243, Val Macro F1=0.855263

Checkpoint history verification: PASS
Ready to resume from Epoch 3.


In [62]:
# ============================================================
# RESTORE TRAINING CONFIGURATION
# ============================================================

criterion = nn.CrossEntropyLoss()

LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
NUM_EPOCHS = 5
PATIENCE = 2

print("=" * 70)
print("TRAINING CONFIGURATION RESTORED")
print("=" * 70)

print("Loss function :", "CrossEntropyLoss")
print("Learning rate :", LEARNING_RATE)
print("Weight decay  :", WEIGHT_DECAY)
print("Total epochs  :", NUM_EPOCHS)
print("Start epoch   :", start_epoch)
print("Patience      :", PATIENCE)

assert start_epoch == 3
assert NUM_EPOCHS == 5
assert LEARNING_RATE == 0.001
assert WEIGHT_DECAY == 0.0001

print("\nConfiguration verification: PASS")

TRAINING CONFIGURATION RESTORED
Loss function : CrossEntropyLoss
Learning rate : 0.001
Weight decay  : 0.0001
Total epochs  : 5
Start epoch   : 3
Patience      : 2

Configuration verification: PASS


In [63]:
# ============================================================
# TRAINING + VALIDATION FUNCTIONS
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_labels = []
    all_predictions = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.detach().cpu().numpy())
        all_predictions.extend(predictions.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    epoch_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    epoch_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return epoch_loss, epoch_accuracy, epoch_macro_f1


@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_labels = []
    all_predictions = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.detach().cpu().numpy())
        all_predictions.extend(predictions.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    epoch_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    epoch_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return epoch_loss, epoch_accuracy, epoch_macro_f1


print("=" * 70)
print("TRAINING FUNCTIONS READY")
print("=" * 70)

print("train_one_epoch : PASS")
print("evaluate_model  : PASS")
print("No training performed.")

TRAINING FUNCTIONS READY
train_one_epoch : PASS
evaluate_model  : PASS
No training performed.


In [64]:
# ============================================================
# KAGGLE OUTPUT DIRECTORIES
# ============================================================

OUTPUT_DIR = "/kaggle/working/corruption_aware_training"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_OUTPUT_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best_model.pth"
)

LATEST_OUTPUT_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "latest_checkpoint.pth"
)

print("=" * 70)
print("OUTPUT DIRECTORIES READY")
print("=" * 70)

print("Output directory:")
print(OUTPUT_DIR)

print("\nCheckpoint directory:")
print(CHECKPOINT_DIR)

print("\nBest checkpoint:")
print(BEST_OUTPUT_CHECKPOINT)

print("\nLatest checkpoint:")
print(LATEST_OUTPUT_CHECKPOINT)

print("\nDirectory verification: PASS")

OUTPUT DIRECTORIES READY
Output directory:
/kaggle/working/corruption_aware_training

Checkpoint directory:
/kaggle/working/corruption_aware_training/checkpoints

Best checkpoint:
/kaggle/working/corruption_aware_training/checkpoints/best_model.pth

Latest checkpoint:
/kaggle/working/corruption_aware_training/checkpoints/latest_checkpoint.pth

Directory verification: PASS


In [66]:
# ============================================================
# DIAGNOSE DATASET FILENAME MISMATCH
# ============================================================

import os
import re

all_records = (
    train_records +
    val_records +
    test_records
)

missing = []
recoverable = []

for record in all_records:
    path = record["path"]

    if os.path.exists(path):
        continue

    missing.append(path)

    # Candidate: remove whitespace immediately before file extension
    candidate = re.sub(r"\s+(\.[^.]+)$", r"\1", path)

    if os.path.exists(candidate):
        recoverable.append((path, candidate))

print("=" * 70)
print("DATASET PATH AUDIT")
print("=" * 70)

print(f"Total records checked : {len(all_records):,}")
print(f"Missing files         : {len(missing):,}")
print(f"Recoverable files     : {len(recoverable):,}")

if recoverable:
    print("\nRecoverable filename mismatches:")
    for old_path, new_path in recoverable:
        print("\nOLD:")
        print(old_path)
        print("NEW:")
        print(new_path)

print("\n" + "=" * 70)

if len(missing) == len(recoverable):
    print("Diagnosis: PASS")
    print("All missing paths have an existing corrected filename.")
else:
    print("Diagnosis: CHECK REQUIRED")
    print("Some missing paths do not have the expected corrected filename.")

DATASET PATH AUDIT
Total records checked : 54,305
Missing files         : 6
Recoverable files     : 6

Recoverable filename mismatches:

OLD:
/kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Corn_(maize)___healthy/af647b0e-db3a-459e-915c-4e522cb46c6c___R.S_HL 8101 copy .jpg
NEW:
/kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Corn_(maize)___healthy/af647b0e-db3a-459e-915c-4e522cb46c6c___R.S_HL 8101 copy.jpg

OLD:
/kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Corn_(maize)___healthy/116ba4e1-7cd5-4d4e-b876-5212a41a0ab3___R.S_HL 8201 copy .jpg
NEW:
/kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Corn_(maize)___healthy/116ba4e1-7cd5-4d4e-b876-5212a41a0ab3___R.S_HL 8201 copy.jpg

OLD:
/kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Tomato___Late_blight/c9f19d2a-8c9d-4ac1-b077-545c5d6f53a9___GHLB_PS Leaf 53 Day 18 .jpg
NEW:
/kaggle/input/datasets/mahmudnadim/robu

In [67]:
# ============================================================
# FIX SPLIT PATH RESOLUTION — NON-DESTRUCTIVE
# ============================================================

def resolve_existing_image_path(image_path):
    """
    Resolve a split-referenced image path without modifying
    the underlying dataset or split files.

    First tries the exact path.
    If missing, removes whitespace immediately before the
    file extension and tries again.
    """
    if os.path.exists(image_path):
        return image_path

    corrected_path = re.sub(
        r"\s+(\.[^.]+)$",
        r"\1",
        image_path
    )

    if os.path.exists(corrected_path):
        return corrected_path

    raise FileNotFoundError(
        f"Image could not be resolved:\n{image_path}"
    )


# Update only the path stored in the record dictionaries.
# Labels and split membership remain unchanged.

for records in [train_records, val_records, test_records]:
    for record in records:
        record["path"] = resolve_existing_image_path(record["path"])


# Rebuild DataFrames from the corrected paths
train_df = pd.DataFrame(train_records)
val_df = pd.DataFrame(val_records)
test_df = pd.DataFrame(test_records)

print("=" * 70)
print("PATH RESOLUTION FIXED")
print("=" * 70)

print(f"Train records : {len(train_df):,}")
print(f"Val records   : {len(val_df):,}")
print(f"Test records  : {len(test_df):,}")

missing_after_fix = sum(
    not os.path.exists(path)
    for path in (
        train_df["path"].tolist()
        + val_df["path"].tolist()
        + test_df["path"].tolist()
    )
)

print(f"Missing files : {missing_after_fix:,}")

assert len(train_df) == 39207
assert len(val_df) == 4389
assert len(test_df) == 10709
assert missing_after_fix == 0

print("\nPath resolution verification: PASS")
print("Split membership and labels were not changed.")

PATH RESOLUTION FIXED
Train records : 39,207
Val records   : 4,389
Test records  : 10,709
Missing files : 0

Path resolution verification: PASS
Split membership and labels were not changed.


In [68]:
# ============================================================
# RESTORE EPOCH-2 CHECKPOINT AFTER DATASET PATH FIX
# ============================================================

checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=DEVICE
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"] + 1
best_val_macro_f1 = checkpoint["best_val_macro_f1"]
best_epoch = checkpoint["best_epoch"]
early_stop_counter = checkpoint["early_stop_counter"]
history = checkpoint["history"]

print("=" * 70)
print("EPOCH-2 CHECKPOINT RESTORED")
print("=" * 70)

print(f"Checkpoint epoch : {checkpoint['epoch']}")
print(f"Next epoch      : {start_epoch}")
print(f"Best epoch      : {best_epoch}")
print(f"Best Val Macro F1 : {best_val_macro_f1:.6f}")
print(f"History epochs   : {len(history['train_loss'])}")
print(f"Early-stop count : {early_stop_counter}")

assert checkpoint["epoch"] == 2
assert start_epoch == 3
assert len(history["train_loss"]) == 2
assert abs(best_val_macro_f1 - 0.8552633281812322) < 1e-8

print("\nCheckpoint restoration: PASS")
print("Ready to restart Epoch 3.")

EPOCH-2 CHECKPOINT RESTORED
Checkpoint epoch : 2
Next epoch      : 3
Best epoch      : 2
Best Val Macro F1 : 0.855263
History epochs   : 2
Early-stop count : 0

Checkpoint restoration: PASS
Ready to restart Epoch 3.


In [71]:
# ============================================================
# REBUILD DATASETS + DATALOADERS USING CORRECTED PATHS
# ============================================================

# Recreate datasets from the corrected DataFrames
corruption_train_dataset = PlantVillageDataset(
    train_df,
    transform=corruption_aware_train_transform
)

corruption_val_dataset = PlantVillageDataset(
    val_df,
    transform=corruption_aware_eval_transform
)

corruption_test_dataset = PlantVillageDataset(
    test_df,
    transform=corruption_aware_eval_transform
)

# Recreate DataLoaders
corruption_train_loader = DataLoader(
    corruption_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

corruption_val_loader = DataLoader(
    corruption_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

corruption_test_loader = DataLoader(
    corruption_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

print("=" * 70)
print("DATASETS + DATALOADERS REBUILT")
print("=" * 70)

print(f"Train dataset : {len(corruption_train_dataset):,}")
print(f"Val dataset   : {len(corruption_val_dataset):,}")
print(f"Test dataset  : {len(corruption_test_dataset):,}")

print(f"\nTrain batches : {len(corruption_train_loader):,}")
print(f"Val batches   : {len(corruption_val_loader):,}")
print(f"Test batches  : {len(corruption_test_loader):,}")

assert len(corruption_train_dataset) == 39207
assert len(corruption_val_dataset) == 4389
assert len(corruption_test_dataset) == 10709

print("\nDataset/DataLoader rebuild: PASS")

DATASETS + DATALOADERS REBUILT
Train dataset : 39,207
Val dataset   : 4,389
Test dataset  : 10,709

Train batches : 1,226
Val batches   : 138
Test batches  : 335

Dataset/DataLoader rebuild: PASS


In [72]:
# ============================================================
# DIRECT READ TEST — PREVIOUSLY FAILING IMAGE
# ============================================================

test_filename = "116ba4e1-7cd5-4d4e-b876-5212a41a0ab3___R.S_HL 8201 copy .jpg"

matching_rows = train_df[
    train_df["path"].str.contains(
        "116ba4e1-7cd5-4d4e-b876-5212a41a0ab3",
        regex=False
    )
]

assert len(matching_rows) == 1

test_index = matching_rows.index[0]

# Read through the actual Dataset object
image, label = corruption_train_dataset.iloc[test_index] if False else corruption_train_dataset[test_index]

print("=" * 70)
print("DIRECT IMAGE READ TEST")
print("=" * 70)

print("Original split filename:")
print(test_filename)

print("\nResolved path:")
print(train_df.iloc[test_index]["path"])

print("\nImage shape:")
print(tuple(image.shape))

print("Image dtype:")
print(image.dtype)

print("Label:")
print(label)

assert image.shape == (3, 224, 224)
assert os.path.exists(train_df.iloc[test_index]["path"])

print("\nPreviously failing image read: PASS")

DIRECT IMAGE READ TEST
Original split filename:
116ba4e1-7cd5-4d4e-b876-5212a41a0ab3___R.S_HL 8201 copy .jpg

Resolved path:
/kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage/color/Corn_(maize)___healthy/116ba4e1-7cd5-4d4e-b876-5212a41a0ab3___R.S_HL 8201 copy.jpg

Image shape:
(3, 224, 224)
Image dtype:
torch.float32
Label:
10

Previously failing image read: PASS


In [73]:
# ============================================================
# CORRUPTION-AWARE TRAINING — EPOCH 3 → EPOCH 5
# ============================================================

import time
from tqdm.auto import tqdm

print("=" * 80)
print("STARTING CORRUPTION-AWARE TRAINING")
print("=" * 80)
print(f"Starting epoch : {start_epoch}")
print(f"Final epoch    : {NUM_EPOCHS}")
print(f"Best epoch     : {best_epoch}")
print(f"Best Val F1    : {best_val_macro_f1:.6f}")
print("=" * 80)

for epoch in range(start_epoch, NUM_EPOCHS + 1):

    epoch_start_time = time.time()

    print(f"\n{'=' * 80}")
    print(f"EPOCH {epoch}/{NUM_EPOCHS}")
    print(f"{'=' * 80}")

    model.train()

    running_loss = 0.0
    all_train_labels = []
    all_train_predictions = []

    train_progress = tqdm(
        corruption_train_loader,
        total=len(corruption_train_loader),
        desc=f"Epoch {epoch} Training",
        leave=True
    )

    for images, labels in train_progress:

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_train_labels.extend(
            labels.detach().cpu().numpy()
        )
        all_train_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        train_progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = running_loss / len(corruption_train_loader.dataset)

    train_accuracy = accuracy_score(
        all_train_labels,
        all_train_predictions
    )

    train_macro_f1 = f1_score(
        all_train_labels,
        all_train_predictions,
        average="macro",
        zero_division=0
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_loss, val_accuracy, val_macro_f1 = evaluate_model(
        model,
        corruption_val_loader,
        criterion,
        DEVICE
    )

    # --------------------------------------------------------
    # UPDATE HISTORY
    # --------------------------------------------------------

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_accuracy"].append(val_accuracy)
    history["train_macro_f1"].append(train_macro_f1)
    history["val_macro_f1"].append(val_macro_f1)

    # --------------------------------------------------------
    # BEST MODEL / EARLY STOPPING
    # --------------------------------------------------------

    is_best = val_macro_f1 > best_val_macro_f1

    if is_best:
        best_val_macro_f1 = val_macro_f1
        best_epoch = epoch
        early_stop_counter = 0
    else:
        early_stop_counter += 1

    epoch_time = time.time() - epoch_start_time

    # --------------------------------------------------------
    # SAVE CHECKPOINT
    # --------------------------------------------------------

    checkpoint_state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "best_val_macro_f1": best_val_macro_f1,
        "best_epoch": best_epoch,
        "early_stop_counter": early_stop_counter,
        "experiment_config": {
            "dataset": "PlantVillage",
            "num_classes": NUM_CLASSES,
            "model": "EfficientNetV2-S",
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "loss": "CrossEntropyLoss",
            "corruption_probability": CORRUPTION_PROBABILITY,
            "corruption_types": CORRUPTION_TYPES,
            "total_epochs": NUM_EPOCHS,
            "patience": PATIENCE,
            "seed": SEED
        }
    }

    torch.save(
        checkpoint_state,
        LATEST_OUTPUT_CHECKPOINT
    )

    if is_best:
        torch.save(
            checkpoint_state,
            BEST_OUTPUT_CHECKPOINT
        )

    # --------------------------------------------------------
    # EPOCH SUMMARY
    # --------------------------------------------------------

    print("\n" + "-" * 80)
    print(f"Epoch {epoch} completed in {epoch_time / 60:.2f} minutes")
    print("-" * 80)

    print(f"Train Loss       : {train_loss:.6f}")
    print(f"Val Loss         : {val_loss:.6f}")
    print(f"Train Accuracy   : {train_accuracy:.6f}")
    print(f"Val Accuracy     : {val_accuracy:.6f}")
    print(f"Train Macro F1   : {train_macro_f1:.6f}")
    print(f"Val Macro F1     : {val_macro_f1:.6f}")
    print(f"Best Val F1      : {best_val_macro_f1:.6f}")
    print(f"Best Epoch       : {best_epoch}")
    print(f"Early Stop       : {early_stop_counter}/{PATIENCE}")
    print(f"Best Updated     : {'YES' if is_best else 'NO'}")

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if early_stop_counter >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\n" + "=" * 80)
print("CORRUPTION-AWARE TRAINING FINISHED")
print("=" * 80)
print(f"Best Epoch        : {best_epoch}")
print(f"Best Val Macro F1 : {best_val_macro_f1:.6f}")
print(f"Epochs in history : {len(history['train_loss'])}")
print("=" * 80)

STARTING CORRUPTION-AWARE TRAINING
Starting epoch : 3
Final epoch    : 5
Best epoch     : 2
Best Val F1    : 0.855263

EPOCH 3/5


Epoch 3 Training:   0%|          | 0/1226 [00:00<?, ?it/s]


--------------------------------------------------------------------------------
Epoch 3 completed in 5.41 minutes
--------------------------------------------------------------------------------
Train Loss       : 0.484650
Val Loss         : 0.383418
Train Accuracy   : 0.854210
Val Accuracy     : 0.888357
Train Macro F1   : 0.823743
Val Macro F1     : 0.863931
Best Val F1      : 0.863931
Best Epoch       : 3
Early Stop       : 0/2
Best Updated     : YES

EPOCH 4/5


Epoch 4 Training:   0%|          | 0/1226 [00:00<?, ?it/s]


--------------------------------------------------------------------------------
Epoch 4 completed in 4.16 minutes
--------------------------------------------------------------------------------
Train Loss       : 0.464394
Val Loss         : 0.333799
Train Accuracy   : 0.855306
Val Accuracy     : 0.903395
Train Macro F1   : 0.829321
Val Macro F1     : 0.879525
Best Val F1      : 0.879525
Best Epoch       : 4
Early Stop       : 0/2
Best Updated     : YES

EPOCH 5/5


Epoch 5 Training:   0%|          | 0/1226 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ebfda29f6a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ebfda29f6a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16


--------------------------------------------------------------------------------
Epoch 5 completed in 4.16 minutes
--------------------------------------------------------------------------------
Train Loss       : 0.453350
Val Loss         : 0.335016
Train Accuracy   : 0.861555
Val Accuracy     : 0.903167
Train Macro F1   : 0.836551
Val Macro F1     : 0.879392
Best Val F1      : 0.879525
Best Epoch       : 4
Early Stop       : 1/2
Best Updated     : NO

CORRUPTION-AWARE TRAINING FINISHED
Best Epoch        : 4
Best Val Macro F1 : 0.879525
Epochs in history : 5


In [75]:
# ============================================================
# VERIFY BEST CHECKPOINT — CORRECTED
# ============================================================

print("=" * 70)
print("BEST CHECKPOINT VERIFICATION")
print("=" * 70)

assert os.path.exists(BEST_OUTPUT_CHECKPOINT)

best_checkpoint = torch.load(
    BEST_OUTPUT_CHECKPOINT,
    map_location="cpu"
)

checkpoint_history = best_checkpoint["history"]

print(f"Checkpoint epoch    : {best_checkpoint['epoch']}")
print(f"Best epoch          : {best_checkpoint['best_epoch']}")
print(
    f"Best Val Macro F1   : "
    f"{best_checkpoint['best_val_macro_f1']:.6f}"
)
print(
    f"Checkpoint history  : "
    f"{len(checkpoint_history['train_loss'])} epochs"
)
print(
    f"Full training history: "
    f"{len(history['train_loss'])} epochs"
)

assert best_checkpoint["epoch"] == 4
assert best_checkpoint["best_epoch"] == 4
assert abs(
    best_checkpoint["best_val_macro_f1"] - 0.879525
) < 1e-6

assert len(checkpoint_history["train_loss"]) == 4
assert len(history["train_loss"]) == 5

print("\nBest checkpoint verification: PASS")
print("Epoch 4 is confirmed as the best model.")

BEST CHECKPOINT VERIFICATION
Checkpoint epoch    : 4
Best epoch          : 4
Best Val Macro F1   : 0.879525
Checkpoint history  : 4 epochs
Full training history: 5 epochs

Best checkpoint verification: PASS
Epoch 4 is confirmed as the best model.


In [76]:
# ============================================================
# LOAD BEST CORRUPTION-AWARE MODEL — EPOCH 4
# ============================================================

best_checkpoint = torch.load(
    BEST_OUTPUT_CHECKPOINT,
    map_location=DEVICE
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.eval()

print("=" * 70)
print("BEST CORRUPTION-AWARE MODEL LOADED")
print("=" * 70)

print(f"Checkpoint epoch    : {best_checkpoint['epoch']}")
print(f"Best epoch          : {best_checkpoint['best_epoch']}")
print(
    f"Best Val Macro F1   : "
    f"{best_checkpoint['best_val_macro_f1']:.6f}"
)
print(f"Device              : {DEVICE}")

# Verify a forward pass
sample_images, sample_labels = next(
    iter(corruption_test_loader)
)

sample_images = sample_images.to(DEVICE)

with torch.no_grad():
    sample_outputs = model(sample_images)

print("\nForward-pass verification:")
print(f"Input shape         : {tuple(sample_images.shape)}")
print(f"Output shape        : {tuple(sample_outputs.shape)}")

assert best_checkpoint["epoch"] == 4
assert sample_outputs.shape == (
    sample_images.shape[0],
    NUM_CLASSES
)

print("\nModel loading verification: PASS")

BEST CORRUPTION-AWARE MODEL LOADED
Checkpoint epoch    : 4
Best epoch          : 4
Best Val Macro F1   : 0.879525
Device              : cuda

Forward-pass verification:
Input shape         : (32, 3, 224, 224)
Output shape        : (32, 38)

Model loading verification: PASS


In [77]:
# ============================================================
# CLEAN TEST EVALUATION — CORRUPTION-AWARE MODEL
# ============================================================

model.eval()

test_loss = 0.0
all_test_labels = []
all_test_predictions = []

with torch.no_grad():

    test_progress = tqdm(
        corruption_test_loader,
        total=len(corruption_test_loader),
        desc="Clean Test Evaluation",
        leave=True
    )

    for images, labels in test_progress:

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_test_labels.extend(
            labels.cpu().numpy()
        )
        all_test_predictions.extend(
            predictions.cpu().numpy()
        )

test_loss /= len(corruption_test_loader.dataset)

test_accuracy = accuracy_score(
    all_test_labels,
    all_test_predictions
)

test_macro_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="weighted",
    zero_division=0
)

test_macro_precision = precision_score(
    all_test_labels,
    all_test_predictions,
    average="macro",
    zero_division=0
)

test_macro_recall = recall_score(
    all_test_labels,
    all_test_predictions,
    average="macro",
    zero_division=0
)

print("\n" + "=" * 70)
print("CORRUPTION-AWARE MODEL — CLEAN TEST RESULTS")
print("=" * 70)

print(f"Test Loss        : {test_loss:.6f}")
print(f"Test Accuracy    : {test_accuracy * 100:.2f}%")
print(f"Macro F1         : {test_macro_f1 * 100:.2f}%")
print(f"Weighted F1      : {test_weighted_f1 * 100:.2f}%")
print(f"Macro Precision  : {test_macro_precision * 100:.2f}%")
print(f"Macro Recall     : {test_macro_recall * 100:.2f}%")
print(f"Correct          : {sum(np.array(all_test_labels) == np.array(all_test_predictions)):,}")
print(f"Incorrect        : {sum(np.array(all_test_labels) != np.array(all_test_predictions)):,}")
print("=" * 70)

Clean Test Evaluation:   0%|          | 0/335 [00:00<?, ?it/s]


CORRUPTION-AWARE MODEL — CLEAN TEST RESULTS
Test Loss        : 0.342374
Test Accuracy    : 89.82%
Macro F1         : 87.35%
Weighted F1      : 89.63%
Macro Precision  : 87.37%
Macro Recall     : 87.77%
Correct          : 9,619
Incorrect        : 1,090


In [78]:
# ============================================================
# CORRUPTION-AWARE MODEL — ROBUSTNESS EVALUATION
# ============================================================

from PIL import ImageEnhance, ImageFilter
import random
import numpy as np

# ------------------------------------------------------------
# Evaluation corruption functions
# Same severity definitions used for baseline evaluation
# ------------------------------------------------------------

def eval_brightness(image, factor):
    return ImageEnhance.Brightness(image).enhance(factor)


def eval_gaussian_noise(image, std):
    image_array = np.asarray(image).astype(np.float32) / 255.0
    noise = np.random.normal(
        0.0,
        std,
        size=image_array.shape
    )
    noisy = np.clip(
        image_array + noise,
        0.0,
        1.0
    )
    noisy = (noisy * 255.0).astype(np.uint8)
    return Image.fromarray(noisy)


def eval_gaussian_blur(image, radius):
    return image.filter(
        ImageFilter.GaussianBlur(radius=radius)
    )


# ------------------------------------------------------------
# Evaluation transform factory
# ------------------------------------------------------------

def make_corruption_transform(corruption_type, severity):

    def apply_corruption(image):

        if corruption_type == "brightness":
            return eval_brightness(image, severity)

        elif corruption_type == "gaussian_noise":
            return eval_gaussian_noise(image, severity)

        elif corruption_type == "gaussian_blur":
            return eval_gaussian_blur(image, severity)

        return image

    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Lambda(apply_corruption),
        transforms.ToTensor(),
        transforms.Normalize(
            IMAGENET_MEAN,
            IMAGENET_STD
        )
    ])


# ------------------------------------------------------------
# Corruption configurations
# ------------------------------------------------------------

ROBUSTNESS_CONFIG = {
    "brightness": {
        "mild": 0.80,
        "moderate": 0.60,
        "severe": 0.40
    },
    "gaussian_noise": {
        "mild": 0.03,
        "moderate": 0.07,
        "severe": 0.12
    },
    "gaussian_blur": {
        "mild": 1,
        "moderate": 2,
        "severe": 3
    }
}

# ------------------------------------------------------------
# Evaluate one corruption condition
# ------------------------------------------------------------

def evaluate_corruption(
    model,
    dataframe,
    corruption_type,
    severity,
    batch_size=32
):

    transform = make_corruption_transform(
        corruption_type,
        severity
    )

    dataset = PlantVillageDataset(
        dataframe,
        transform=transform
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        drop_last=False
    )

    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            outputs = model(images)

            predictions = outputs.argmax(
                dim=1
            )

            all_labels.extend(
                labels.numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return accuracy, macro_f1


print("=" * 70)
print("ROBUSTNESS EVALUATION CONFIGURATION READY")
print("=" * 70)

for corruption, severities in ROBUSTNESS_CONFIG.items():
    print(f"{corruption}: {list(severities.keys())}")

print("\nNo robustness evaluation performed yet.")

ROBUSTNESS EVALUATION CONFIGURATION READY
brightness: ['mild', 'moderate', 'severe']
gaussian_noise: ['mild', 'moderate', 'severe']
gaussian_blur: ['mild', 'moderate', 'severe']

No robustness evaluation performed yet.


In [79]:
# ============================================================
# RUN ROBUSTNESS EVALUATION — 9 CONDITIONS
# ============================================================

robustness_results = []

for corruption_type, severities in ROBUSTNESS_CONFIG.items():

    for severity_name, severity_value in severities.items():

        print(
            f"\nEvaluating: "
            f"{corruption_type} | {severity_name}"
        )

        accuracy, macro_f1 = evaluate_corruption(
            model=model,
            dataframe=test_df,
            corruption_type=corruption_type,
            severity=severity_value,
            batch_size=BATCH_SIZE
        )

        robustness_results.append({
            "corruption": corruption_type,
            "severity": severity_name,
            "severity_value": severity_value,
            "accuracy": accuracy,
            "macro_f1": macro_f1
        })

        print(
            f"Accuracy: {accuracy * 100:.2f}% | "
            f"Macro F1: {macro_f1 * 100:.2f}%"
        )

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

robustness_df = pd.DataFrame(robustness_results)

print("\n" + "=" * 80)
print("CORRUPTION-AWARE MODEL — ROBUSTNESS RESULTS")
print("=" * 80)

display(
    robustness_df.style.format({
        "accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
        "severity_value": "{:.3f}"
    })
)

print("\nRobustness evaluation completed successfully.")


Evaluating: brightness | mild
Accuracy: 89.39% | Macro F1: 86.91%

Evaluating: brightness | moderate
Accuracy: 87.87% | Macro F1: 85.23%

Evaluating: brightness | severe
Accuracy: 83.04% | Macro F1: 78.89%

Evaluating: gaussian_noise | mild
Accuracy: 88.79% | Macro F1: 86.19%

Evaluating: gaussian_noise | moderate
Accuracy: 81.57% | Macro F1: 77.18%

Evaluating: gaussian_noise | severe
Accuracy: 68.11% | Macro F1: 62.21%

Evaluating: gaussian_blur | mild
Accuracy: 84.13% | Macro F1: 80.70%

Evaluating: gaussian_blur | moderate
Accuracy: 66.32% | Macro F1: 60.99%

Evaluating: gaussian_blur | severe
Accuracy: 43.09% | Macro F1: 35.19%

CORRUPTION-AWARE MODEL — ROBUSTNESS RESULTS


,corruption,severity,severity_value,accuracy,macro_f1
0,brightness,mild,0.800,0.8939,0.8691
1,brightness,moderate,0.600,0.8787,0.8523
2,brightness,severe,0.400,0.8304,0.7889
3,gaussian_noise,mild,0.030,0.8879,0.8619
4,gaussian_noise,moderate,0.070,0.8157,0.7718
5,gaussian_noise,severe,0.120,0.6811,0.6221
6,gaussian_blur,mild,1.000,0.8413,0.8070
7,gaussian_blur,moderate,2.000,0.6632,0.6099
8,gaussian_blur,severe,3.000,0.4309,0.3519



Robustness evaluation completed successfully.


In [80]:
# ============================================================
# BASELINE vs CORRUPTION-AWARE ROBUSTNESS COMPARISON
# ============================================================

baseline_results = [
    {"corruption": "brightness", "severity": "mild",
     "accuracy": 0.8962, "macro_f1": 0.8738},
    {"corruption": "brightness", "severity": "moderate",
     "accuracy": 0.8806, "macro_f1": 0.8556},
    {"corruption": "brightness", "severity": "severe",
     "accuracy": 0.8323, "macro_f1": 0.7968},

    {"corruption": "gaussian_noise", "severity": "mild",
     "accuracy": 0.8823, "macro_f1": 0.8580},
    {"corruption": "gaussian_noise", "severity": "moderate",
     "accuracy": 0.8299, "macro_f1": 0.7952},
    {"corruption": "gaussian_noise", "severity": "severe",
     "accuracy": 0.7243, "macro_f1": 0.6782},

    {"corruption": "gaussian_blur", "severity": "mild",
     "accuracy": 0.8078, "macro_f1": 0.7769},
    {"corruption": "gaussian_blur", "severity": "moderate",
     "accuracy": 0.5909, "macro_f1": 0.5642},
    {"corruption": "gaussian_blur", "severity": "severe",
     "accuracy": 0.3638, "macro_f1": 0.3207},
]

baseline_df = pd.DataFrame(baseline_results)

comparison_df = baseline_df.merge(
    robustness_df[
        ["corruption", "severity", "accuracy", "macro_f1"]
    ],
    on=["corruption", "severity"],
    suffixes=("_baseline", "_corruption_aware")
)

comparison_df["accuracy_change_pp"] = (
    comparison_df["accuracy_corruption_aware"]
    - comparison_df["accuracy_baseline"]
) * 100

comparison_df["macro_f1_change_pp"] = (
    comparison_df["macro_f1_corruption_aware"]
    - comparison_df["macro_f1_baseline"]
) * 100

print("=" * 100)
print("BASELINE vs CORRUPTION-AWARE ROBUSTNESS")
print("=" * 100)

display(
    comparison_df.style.format({
        "accuracy_baseline": "{:.2%}",
        "accuracy_corruption_aware": "{:.2%}",
        "macro_f1_baseline": "{:.2%}",
        "macro_f1_corruption_aware": "{:.2%}",
        "accuracy_change_pp": "{:+.2f}",
        "macro_f1_change_pp": "{:+.2f}",
    })
)

print("\nOverall mean performance:")

print(
    f"Baseline mean Accuracy: "
    f"{comparison_df['accuracy_baseline'].mean() * 100:.2f}%"
)

print(
    f"Corruption-aware mean Accuracy: "
    f"{comparison_df['accuracy_corruption_aware'].mean() * 100:.2f}%"
)

print(
    f"Mean Accuracy change: "
    f"{comparison_df['accuracy_change_pp'].mean():+.2f} pp"
)

print(
    f"\nBaseline mean Macro F1: "
    f"{comparison_df['macro_f1_baseline'].mean() * 100:.2f}%"
)

print(
    f"Corruption-aware mean Macro F1: "
    f"{comparison_df['macro_f1_corruption_aware'].mean() * 100:.2f}%"
)

print(
    f"Mean Macro F1 change: "
    f"{comparison_df['macro_f1_change_pp'].mean():+.2f} pp"
)

print("\nComparison calculation: PASS")

BASELINE vs CORRUPTION-AWARE ROBUSTNESS


,corruption,severity,accuracy_baseline,macro_f1_baseline,accuracy_corruption_aware,macro_f1_corruption_aware,accuracy_change_pp,macro_f1_change_pp
0,brightness,mild,89.62%,87.38%,89.39%,86.91%,-0.23,-0.47
1,brightness,moderate,88.06%,85.56%,87.87%,85.23%,-0.19,-0.33
2,brightness,severe,83.23%,79.68%,83.04%,78.89%,-0.19,-0.79
3,gaussian_noise,mild,88.23%,85.80%,88.79%,86.19%,+0.56,+0.39
4,gaussian_noise,moderate,82.99%,79.52%,81.57%,77.18%,-1.42,-2.34
5,gaussian_noise,severe,72.43%,67.82%,68.11%,62.21%,-4.32,-5.61
6,gaussian_blur,mild,80.78%,77.69%,84.13%,80.70%,+3.35,+3.01
7,gaussian_blur,moderate,59.09%,56.42%,66.32%,60.99%,+7.23,+4.57
8,gaussian_blur,severe,36.38%,32.07%,43.09%,35.19%,+6.71,+3.12



Overall mean performance:
Baseline mean Accuracy: 75.65%
Corruption-aware mean Accuracy: 76.92%
Mean Accuracy change: +1.28 pp

Baseline mean Macro F1: 72.44%
Corruption-aware mean Macro F1: 72.61%
Mean Macro F1 change: +0.17 pp

Comparison calculation: PASS


In [81]:
# ============================================================
# ROBUSTNESS DROP FROM CLEAN PERFORMANCE
# ============================================================

BASELINE_CLEAN_ACC = 0.9016
BASELINE_CLEAN_F1 = 0.8800

CA_CLEAN_ACC = 0.8982
CA_CLEAN_F1 = 0.8735

comparison_df["baseline_accuracy_drop_pp"] = (
    BASELINE_CLEAN_ACC
    - comparison_df["accuracy_baseline"]
) * 100

comparison_df["ca_accuracy_drop_pp"] = (
    CA_CLEAN_ACC
    - comparison_df["accuracy_corruption_aware"]
) * 100

comparison_df["baseline_f1_drop_pp"] = (
    BASELINE_CLEAN_F1
    - comparison_df["macro_f1_baseline"]
) * 100

comparison_df["ca_f1_drop_pp"] = (
    CA_CLEAN_F1
    - comparison_df["macro_f1_corruption_aware"]
) * 100

comparison_df["baseline_accuracy_retention_pct"] = (
    comparison_df["accuracy_baseline"]
    / BASELINE_CLEAN_ACC
) * 100

comparison_df["ca_accuracy_retention_pct"] = (
    comparison_df["accuracy_corruption_aware"]
    / CA_CLEAN_ACC
) * 100

print("=" * 100)
print("ROBUSTNESS DROP AND CLEAN-PERFORMANCE RETENTION")
print("=" * 100)

display(
    comparison_df[
        [
            "corruption",
            "severity",
            "baseline_accuracy_drop_pp",
            "ca_accuracy_drop_pp",
            "baseline_f1_drop_pp",
            "ca_f1_drop_pp",
            "baseline_accuracy_retention_pct",
            "ca_accuracy_retention_pct"
        ]
    ].style.format({
        "baseline_accuracy_drop_pp": "{:.2f}",
        "ca_accuracy_drop_pp": "{:.2f}",
        "baseline_f1_drop_pp": "{:.2f}",
        "ca_f1_drop_pp": "{:.2f}",
        "baseline_accuracy_retention_pct": "{:.2f}%",
        "ca_accuracy_retention_pct": "{:.2f}%"
    })
)

print("\nRetention/drop calculation: PASS")

ROBUSTNESS DROP AND CLEAN-PERFORMANCE RETENTION


,corruption,severity,baseline_accuracy_drop_pp,ca_accuracy_drop_pp,baseline_f1_drop_pp,ca_f1_drop_pp,baseline_accuracy_retention_pct,ca_accuracy_retention_pct
0,brightness,mild,0.54,0.43,0.62,0.44,99.40%,99.52%
1,brightness,moderate,2.10,1.95,2.44,2.12,97.67%,97.83%
2,brightness,severe,6.93,6.78,8.32,8.46,92.31%,92.45%
3,gaussian_noise,mild,1.93,1.03,2.20,1.16,97.86%,98.85%
4,gaussian_noise,moderate,7.17,8.25,8.48,10.17,92.05%,90.81%
5,gaussian_noise,severe,17.73,21.71,20.18,25.14,80.33%,75.83%
6,gaussian_blur,mild,9.38,5.69,10.31,6.65,89.60%,93.66%
7,gaussian_blur,moderate,31.07,23.50,31.58,26.36,65.54%,73.83%
8,gaussian_blur,severe,53.78,46.73,55.93,52.16,40.35%,47.97%



Retention/drop calculation: PASS


In [82]:
# ============================================================
# SAVE CORRUPTION-AWARE EXPERIMENT RESULTS
# ============================================================

import json
import os

RESULTS_DIR = "/kaggle/working/corruption_aware_training/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save robustness comparison
comparison_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "baseline_vs_corruption_aware_robustness.csv"
    ),
    index=False
)

# Save raw corruption-aware robustness results
robustness_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "corruption_aware_robustness_results.csv"
    ),
    index=False
)

# Save clean-test results
clean_results = {
    "model": "EfficientNetV2-S Corruption-Aware",
    "checkpoint_epoch": 4,
    "test_loss": float(test_loss),
    "accuracy": float(test_accuracy),
    "macro_f1": float(test_macro_f1),
    "weighted_f1": float(test_weighted_f1),
    "macro_precision": float(test_macro_precision),
    "macro_recall": float(test_macro_recall),
    "correct": int(sum(
        np.array(all_test_labels) ==
        np.array(all_test_predictions)
    )),
    "incorrect": int(sum(
        np.array(all_test_labels) !=
        np.array(all_test_predictions)
    ))
}

with open(
    os.path.join(
        RESULTS_DIR,
        "corruption_aware_clean_test_results.json"
    ),
    "w"
) as f:
    json.dump(clean_results, f, indent=2)

print("=" * 70)
print("RESULTS SAVED")
print("=" * 70)

print(f"Results directory: {RESULTS_DIR}")

print("\nFiles:")
for filename in sorted(os.listdir(RESULTS_DIR)):
    filepath = os.path.join(RESULTS_DIR, filename)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  {filename} — {size_kb:.2f} KB")

print("\nResults export: PASS")

RESULTS SAVED
Results directory: /kaggle/working/corruption_aware_training/results

Files:
  baseline_vs_corruption_aware_robustness.csv — 2.19 KB
  corruption_aware_clean_test_results.json — 0.33 KB
  corruption_aware_robustness_results.csv — 0.60 KB

Results export: PASS


In [83]:
import os

BEST_CHECKPOINT = (
    "/kaggle/working/corruption_aware_training/"
    "checkpoints/best_model.pth"
)

print("Exists:", os.path.exists(BEST_CHECKPOINT))

if os.path.exists(BEST_CHECKPOINT):
    print(
        "Size:",
        f"{os.path.getsize(BEST_CHECKPOINT) / (1024**2):.2f} MB"
    )

Exists: True
Size: 78.39 MB


In [84]:
# ============================================================
# PREPARE EPOCH-4 CHECKPOINT FOR KAGGLE DATASET
# ============================================================

import os
import shutil

EXPORT_DIR = "/kaggle/working/epoch4_corruption_aware_checkpoint"
os.makedirs(EXPORT_DIR, exist_ok=True)

source_checkpoint = (
    "/kaggle/working/corruption_aware_training/"
    "checkpoints/best_model.pth"
)

target_checkpoint = os.path.join(
    EXPORT_DIR,
    "best_model.pth"
)

shutil.copy2(
    source_checkpoint,
    target_checkpoint
)

print("=" * 70)
print("CHECKPOINT EXPORT READY")
print("=" * 70)

print("Source:")
print(source_checkpoint)

print("\nExport:")
print(target_checkpoint)

print(
    "\nSize:",
    f"{os.path.getsize(target_checkpoint) / (1024**2):.2f} MB"
)

assert os.path.exists(target_checkpoint)
assert os.path.getsize(target_checkpoint) > 70 * 1024 * 1024

print("\nCheckpoint export verification: PASS")

CHECKPOINT EXPORT READY
Source:
/kaggle/working/corruption_aware_training/checkpoints/best_model.pth

Export:
/kaggle/working/epoch4_corruption_aware_checkpoint/best_model.pth

Size: 78.39 MB

Checkpoint export verification: PASS


In [86]:
# ============================================================
# PACKAGE CHECKPOINT FOR DOWNLOAD
# ============================================================

import shutil
import os

checkpoint = (
    "/kaggle/working/epoch4_corruption_aware_checkpoint/"
    "best_model.pth"
)

zip_base = "/kaggle/working/epoch4_corruption_aware_checkpoint"

zip_path = shutil.make_archive(
    zip_base,
    "zip",
    "/kaggle/working",
    "epoch4_corruption_aware_checkpoint"
)

print("=" * 70)
print("CHECKPOINT PACKAGE CREATED")
print("=" * 70)

print("ZIP:")
print(zip_path)

print(
    "Size:",
    f"{os.path.getsize(zip_path) / (1024**2):.2f} MB"
)

assert os.path.exists(zip_path)

print("\nPackage verification: PASS")

CHECKPOINT PACKAGE CREATED
ZIP:
/kaggle/working/epoch4_corruption_aware_checkpoint.zip
Size: 72.70 MB

Package verification: PASS


In [87]:
# ============================================================
# UPLOAD EPOCH-4 CHECKPOINT DIRECTLY TO KAGGLE DATASETS
# ============================================================

import os
import kagglehub

CHECKPOINT_DIR = (
    "/kaggle/working/epoch4_corruption_aware_checkpoint"
)

DATASET_HANDLE = (
    "mahmudnadim/robust-crop-disease-corruption-aware-checkpoint"
)

print("=" * 70)
print("UPLOADING EPOCH-4 CHECKPOINT TO KAGGLE DATASET")
print("=" * 70)

print("Source directory:")
print(CHECKPOINT_DIR)

print("\nDataset:")
print(DATASET_HANDLE)

assert os.path.exists(
    os.path.join(CHECKPOINT_DIR, "best_model.pth")
)

print("\nCheckpoint found: PASS")
print("Starting upload...")

kagglehub.dataset_upload(
    DATASET_HANDLE,
    CHECKPOINT_DIR,
    version_notes="Epoch 4 best corruption-aware EfficientNetV2-S checkpoint"
)

print("\n" + "=" * 70)
print("UPLOAD COMPLETED")
print("=" * 70)

UPLOADING EPOCH-4 CHECKPOINT TO KAGGLE DATASET
Source directory:
/kaggle/working/epoch4_corruption_aware_checkpoint

Dataset:
mahmudnadim/robust-crop-disease-corruption-aware-checkpoint

Checkpoint found: PASS
Starting upload...
Uploading Dataset https://api.kaggle.com/datasets/mahmudnadim/robust-crop-disease-corruption-aware-checkpoint ...
Starting upload for file /kaggle/working/epoch4_corruption_aware_checkpoint/best_model.pth

Uploading: 100%|██████████| 82.2M/82.2M [00:03<00:00, 26.8MB/s]

Upload successful: /kaggle/working/epoch4_corruption_aware_checkpoint/best_model.pth (78MB)


Your dataset has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/mahmudnadim/robust-crop-disease-corruption-aware-checkpoint

UPLOAD COMPLETED
